# VLM-Anomaly — Full MVTec Sweep · Qwen3-VL-32B via OpenRouter

**Model:** `qwen/qwen3-vl-32b-instruct`  
**API:** OpenRouter (OpenAI-compatible)  
**Cost estimate:** ~$0.20 for all 1,245 images  
**Runs locally** against your `.env` `OPEN_ROUTER` key and `data/mvtec` dataset.

In [ ]:
# ── Cell 1: Setup paths & sys.path ─────────────────────────────────────────
import sys
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT   = Path().resolve().parent   # notebooks/ → repo root
SRC_DIR     = REPO_ROOT / 'src'
PROMPTS_DIR = REPO_ROOT / 'prompts'
RESULTS_DIR = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert SRC_DIR.exists(),     f'src/ not found at {SRC_DIR}'
assert PROMPTS_DIR.exists(), f'prompts/ not found at {PROMPTS_DIR}'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(REPO_ROOT / '.env')

import vlm_anomaly
print(f'vlm_anomaly {vlm_anomaly.__version__} ready')
print(f'SRC      : {SRC_DIR}')
print(f'Prompts  : {PROMPTS_DIR}')
print(f'Results  : {RESULTS_DIR}')

In [ ]:
# ── Cell 2: Verify OpenRouter API key ───────────────────────────────────────
import os

api_key = os.environ.get('OPEN_ROUTER', '')
assert api_key, 'OPEN_ROUTER not set — add it to your .env file'
print(f'OPEN_ROUTER: {api_key[:12]}...{api_key[-4:]}')

In [ ]:
# ── Cell 3: Find MVTec dataset ──────────────────────────────────────────────
MVTEC_ROOT = None
for candidate in [
    REPO_ROOT / 'data' / 'mvtec',
    REPO_ROOT / 'data' / 'mvtec-ad',
    Path('/tmp/mvtec'),
]:
    if candidate.exists() and any(candidate.iterdir()):
        MVTEC_ROOT = candidate
        break

assert MVTEC_ROOT, f'MVTec not found. Expected at {REPO_ROOT}/data/mvtec'
categories = sorted([d.name for d in MVTEC_ROOT.iterdir() if d.is_dir()])
print(f'MVTec root : {MVTEC_ROOT}')
print(f'Categories : {len(categories)} → {categories}')

In [ ]:
# ── Cell 4: Configure ───────────────────────────────────────────────────────
MODEL      = 'qwen/qwen3-vl-32b-instruct'
PROMPT_KEY = 'manufacturing.detailed'
LIMIT      = None    # None = all images; set e.g. 5 for a quick smoke test
BUDGET_USD = 10.0    # hard cap — well above the ~$0.20 expected cost

print(f'Model  : {MODEL}')
print(f'Prompt : {PROMPT_KEY}')
print(f'Limit  : {LIMIT or "all images"}')
print(f'Budget : ${BUDGET_USD}')
print(f'Total  : ~{len(categories) * 83} images')

In [ ]:
# ── Cell 5: Build shared objects ────────────────────────────────────────────
from vlm_anomaly.config import Settings
from vlm_anomaly.datasets.mvtec import MVTec
from vlm_anomaly.backends.openrouter import OpenRouterBackend
from vlm_anomaly.evaluators.prompt_library import PromptLibrary
from vlm_anomaly.logging import configure_logging

configure_logging(json_logs=False, log_level='INFO')

settings = Settings(
    _env_file=str(REPO_ROOT / '.env'),
    data_dir=str(MVTEC_ROOT.parent),
    results_dir=str(RESULTS_DIR),
    default_budget_usd=BUDGET_USD,
)
settings.results_dir = RESULTS_DIR

dataset    = MVTec(root_dir=MVTEC_ROOT)
backend    = OpenRouterBackend(model=MODEL)
prompt_lib = PromptLibrary(prompts_dir=PROMPTS_DIR)

print(f'Dataset  : {MVTEC_ROOT}')
print(f'Backend  : {backend.name} / {MODEL}')
print(f'Prompts  : {len(list(prompt_lib._prompts.keys()))} keys loaded')

In [ ]:
# ── Cell 6: Run all 15 categories ───────────────────────────────────────────
from tqdm.auto import tqdm
from vlm_anomaly.schemas import ExperimentConfig
from vlm_anomaly.evaluators.vlm_evaluator import VLMEvaluator

all_results = []
total_cost  = 0.0

for category in tqdm(categories, desc='MVTec categories'):
    existing = [
        f for f in RESULTS_DIR.glob(f'*_mvtec_{category}.jsonl')
        if 'openrouter' in f.name and f.stat().st_size > 100
    ]
    if existing:
        print(f'  [skip] {category} — already done ({existing[0].name})')
        continue

    config = ExperimentConfig(
        backend=f'openrouter/{MODEL}',
        dataset='mvtec',
        categories=[category],
        prompt=PROMPT_KEY,
        limit=LIMIT,
        budget_usd=BUDGET_USD,
    )
    evaluator = VLMEvaluator(
        backend=backend,
        dataset=dataset,
        config=config,
        settings=settings,
        prompt_library=prompt_lib,
    )
    results = evaluator.run()
    all_results.extend(results)
    for r in results:
        cat_cost = sum(
            p.cost_usd for p in evaluator._last_predictions
            if hasattr(evaluator, '_last_predictions')
        ) if hasattr(evaluator, '_last_predictions') else 0.0
        total_cost += cat_cost
        print(
            f'  {category:12s}  AUROC={r.auroc:.3f}  F1={r.f1:.3f}  '
            f'n={r.n_images}  cost=${r.total_cost_usd:.4f}'
        )

print(f'\nDone. {len(all_results)} categories. Estimated total cost: ${sum(r.total_cost_usd for r in all_results):.4f}')

In [ ]:
# ── Cell 7: Leaderboard ─────────────────────────────────────────────────────
from vlm_anomaly.analysis.aggregator import leaderboard, cost_accuracy_table

lb = leaderboard(RESULTS_DIR)
if lb.empty:
    print('No results yet.')
else:
    summary = cost_accuracy_table(RESULTS_DIR)
    print('=== Summary (mean across categories) ===')
    print(summary[['model_id', 'mean_auroc', 'mean_latency_ms']].to_string(index=False))
    print()
    print('=== Per-category breakdown (Qwen3-VL) ===')
    qwen_lb = lb[lb['model_id'].str.contains('openrouter', na=False)]
    display(qwen_lb[['category', 'n_images', 'auroc', 'f1', 'mean_latency_ms']].sort_values('auroc', ascending=False))

In [ ]:
# ── Cell 8: Generate report ──────────────────────────────────────────────────
from vlm_anomaly.analysis.report_generator import generate

report = generate(RESULTS_DIR, str(REPO_ROOT / 'REPORT.md'))
print(f'Report written to {report}')

In [ ]:
# ── Cell 9: Show result files + commit hint ──────────────────────────────────
result_files = sorted(RESULTS_DIR.glob('*openrouter*mvtec*.jsonl'))
print(f'Result files ({len(result_files)}):')
for f in result_files:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name}  ({size_kb:.1f} KB)')

print()
print('To commit results:')
print('  git add results/*.jsonl')
print('  git commit -m "results(qwen3-vl): full MVTec sweep via OpenRouter"')